# Paralelización del método del trapecio

---

## Problema:

Se busca aproximar numéricamente la integral de la función:
$$f(x) = x - \frac{x^{3}}{3!} + \frac{x^{5}}{5!},$$
en el intervalo $[a,b]$, la cual coincide con los primeros tres términos de la serie de Taylor de la función $\sin(x)$. Se usará el método del trapecio para cumplir el objetivo con un número de trapecios $n = 1 \times 10^{7}$.

Luego, se compara la solución numérica con la solución analítica para ver su precisión en la aproximación. La solución analítica es:
$$\int_{a}^{b} f(x) \, \mathrm{d}x = \left(\frac{x^{2}}{2} - \frac{x^{4}}{4!} + \frac{x^{6}}{6!}\right) \Big|_{a}^{b}.$$

## Código secuencial (en serie): `trapecio.cpp`

Usando la función `%%writefile` se genera el archivo para la ejecución secuencial llamado: `trapecio.cpp`.

In [27]:
%%writefile trapecio.cpp
#include <iostream>
#include <cmath>
#include <chrono>

// Función a integrar
double f(double x) {
    return x - pow(x, 3)/6 + pow(x, 5)/120;
}

// Solución analítica
double integral_analitica(double x) {
    return pow(x, 2)/2 - pow(x, 4)/24 + pow(x, 6)/720;
}

// Método del trapecio (secuencial)
double metodo_trapecio(double a, double b, int n) {
    double h = (b - a)/n;
    double suma = (f(a) + f(b))/2.0;
    for (int i = 1; i < n; ++i) {
        suma += f(a + i*h);
    }
    return h * suma;
}

int main() {
    const double a = 0.0, b = 4.0;
    const int n = 10000000;

    // Medición de tiempo
    auto start = std::chrono::high_resolution_clock::now();
    double resultado_numerico = metodo_trapecio(a, b, n);
    auto end = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double> tiempo = end - start;

    // Solución analítica
    double resultado_analitico = integral_analitica(b) - integral_analitica(a);
    double error = fabs((resultado_analitico - resultado_numerico) / resultado_analitico) * 100.0;

    // Resultados
    std::cout << "=== MÉTODO DEL TRAPECIO (SECUENCIAL) ===\n";
    std::cout << "Resultado numérico: " << resultado_numerico << "\n";
    std::cout << "Resultado analítico: " << resultado_analitico << "\n";
    std::cout << "Error relativo: " << error << " %\n";
    std::cout << "Tiempo de ejecución: " << tiempo.count() << " segundos\n";

    return 0;
}

Writing trapecio.cpp


## Código omp (en paralelo): `trapecio_omp.cpp`

Usando la función `%%writefile` se genera el archivo para la ejecución secuencial llamado: `trapecio_omp.cpp`.

In [28]:
%%writefile trapecio_omp.cpp
#include <iostream>
#include <cmath>
#include <chrono>
#include <omp.h>

// Función a integrar
double f(double x) {
    return x - pow(x, 3)/6 + pow(x, 5)/120;
}

// Solución analítica
double integral_analitica(double x) {
    return pow(x, 2)/2 - pow(x, 4)/24 + pow(x, 6)/720;
}

// Método del trapecio paralelizado
double metodo_trapecio_omp(double a, double b, int n) {
    double h = (b - a)/n;
    double suma = (f(a) + f(b))/2.0;
    #pragma omp parallel for reduction(+:suma)
    for (int i = 1; i < n; ++i) {
        suma += f(a + i*h);
    }
    return h * suma;
}

int main() {
    const double a = 0.0, b = 4.0;
    const int n = 10000000;

    // Configuración de hilos
    omp_set_num_threads(omp_get_max_threads());

    // Medición de tiempo
    auto start = std::chrono::high_resolution_clock::now();
    double resultado_numerico = metodo_trapecio_omp(a, b, n);
    auto end = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double> tiempo = end - start;

    // Solución analítica
    double resultado_analitico = integral_analitica(b) - integral_analitica(a);
    double error = fabs((resultado_analitico - resultado_numerico) / resultado_analitico) * 100.0;

    // Resultados
    std::cout << "=== MÉTODO DEL TRAPECIO (OPENMP) ===\n";
    std::cout << "Resultado numérico: " << resultado_numerico << "\n";
    std::cout << "Resultado analítico: " << resultado_analitico << "\n";
    std::cout << "Error relativo: " << error << " %\n";
    std::cout << "Tiempo de ejecución: " << tiempo.count() << " segundos\n";
    std::cout << "Hilos utilizados: " << omp_get_max_threads() << "\n";

    return 0;
}

Writing trapecio_omp.cpp


## Ejecución y resultados

Finalmente, se compilan y ejecutan los códigos imprimiendo en terminal los resultados finales.

In [29]:
# Compilación
!g++ trapecio.cpp -o trapecio
!g++ -fopenmp trapecio_omp.cpp -o trapecio_omp

# Ejecución
print("========================================")
!./trapecio
print("\n====================================")
!./trapecio_omp
print("========================================")

=== MÉTODO DEL TRAPECIO (SECUENCIAL) ===
Resultado numérico: 3.02222
Resultado analítico: 3.02222
Error relativo: 1.43709e-11 %
Tiempo de ejecución: 0.47543 segundos

=== MÉTODO DEL TRAPECIO (OPENMP) ===
Resultado numérico: 3.02222
Resultado analítico: 3.02222
Error relativo: 3.37965e-13 %
Tiempo de ejecución: 0.0910529 segundos
Hilos utilizados: 8


Donde se puede observar que, si bien los resultados numéricos y analíticos son prácticamente idénticos y la precisión es extremadamente alta en ambos casos, el código paralelo presenta una mejora significativa en el rendimiento. En concreto, **el uso de paralelismo con OpenMP reduce el tiempo de ejecución en aproximadamente un 80.85% respecto al enfoque secuencial**. Esto representa un ejemplo claro y concreto de las ventajas que ofrece la programación paralela en problemas computacionalmente intensivos, permitiendo obtener los mismos resultados con una eficiencia mucho mayor.